<a href="https://colab.research.google.com/github/JamesMartinOU/PublicRedditSentimentAnalysis/blob/main/RedditCreateSentimentFactTable_YearWeek.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install Python libraries
!pip install mysql-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.0/34.0 MB 21.8 MB/s eta 0:00:00


In [2]:
# Import Python libraries
import mysql.connector
import pandas as pd
from google.colab import files

In [3]:
# RDS MySQL connection details

In [17]:
# Connect to the MySQL database
conn = mysql.connector.connect(
    host=DB_HOST,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME
)

# Query to create sentiment fact table
query_create_fact_table_weekly = """
CREATE TABLE reddit_sentiment_fact_table_weekly (
  id INT AUTO_INCREMENT PRIMARY KEY,
  type VARCHAR(10),
  company_name VARCHAR(255),
  keyword VARCHAR(255),
  keyword_label VARCHAR(255),
  year INT,
  week INT,
  sentiment VARCHAR(20),
  count INT
);
"""

# Query to generate sentiment fact table data
query_fact_table_sentiment_weekly = """
INSERT INTO
  reddit_sentiment_fact_table_weekly (type, company_name, keyword, keyword_label, year, week, sentiment, count)

SELECT
  "comment" as type,
  rcsgb.company_name,
  rcsgb.keyword,
  CASE
    WHEN rcsgb.keyword = rcsgb.company_name THEN "company_name"
    WHEN length(rcsgb.keyword) <= 5 THEN "stock_symbol"
    ELSE "ceo_name"
  END as keyword_label,
  year(rcsgb.created_utc) as year,
  week(rcsgb.created_utc, 1) as week,
  rcsgb.avg_sentiment as sentiment,
  count(rcsgb.comment_id) as count

FROM (
  SELECT
    arc.post_id,
    arc.comment_id,
    arc.created_utc,
    rp.company_name,
    rp.keyword,
    CASE
      WHEN avg(arc.sentiment_score) >= .33 THEN "Positive"
      WHEN avg(arc.sentiment_score) <= -.33 THEN "Negative"
      ELSE "Neutral"
    END as avg_sentiment
  FROM (
    SELECT
      rcs.post_id,
      rcs.comment_id,
      rc.created_utc,
      CASE
        WHEN rcs.sentiment = "Neutral" THEN 0
        WHEN rcs.sentiment = "Positive" THEN 1
        ELSE -1
      END as sentiment_score
    FROM reddit_comments_sentiment as rcs
    INNER JOIN reddit_comments as rc ON rc.comment_id = rcs.comment_id
  ) as arc
  INNER JOIN
    reddit_posts as rp ON rp.id = arc.post_id
  GROUP BY
    arc.post_id,
    arc.comment_id,
    arc.created_utc,
    rp.company_name,
    rp.keyword
    ) as rcsgb

GROUP BY
  rcsgb.company_name,
  rcsgb.keyword,
  keyword_label,
  year(rcsgb.created_utc),
  week,
  rcsgb.avg_sentiment

UNION ALL

SELECT
  "post" as type,
  rp.Company_Name,
  rp.Keyword,
  CASE
    WHEN rp.keyword = rp.company_name THEN "company_name"
    WHEN length(rp.keyword) <= 5 THEN "stock_symbol"
    ELSE "ceo_name"
  END as keyword_label,
  year(created_utc) as year,
  week(created_utc, 1) as week,
  rps.Sentiment,
  count(rp.id) as count

FROM reddit_posts as rp
INNER JOIN
  reddit_posts_sentiment as rps ON rps.id = rp.id
GROUP BY
  rp.Company_Name,
  rp.Keyword,
  keyword_label,
  year(created_utc),
  week,
  rps.Sentiment
"""


# Create cursor and execute
cursor = conn.cursor()
cursor.execute("DROP TABLE IF EXISTS reddit_sentiment_fact_table_weekly")
cursor.execute(query_create_fact_table_weekly)
cursor.execute(query_fact_table_sentiment_weekly)
conn.commit()


# View results
cursor.execute("SELECT * FROM reddit_sentiment_fact_table_weekly")
columns = [desc[0] for desc in cursor.description]
data = cursor.fetchall()
df_sentiment_fact = pd.DataFrame(data, columns=columns)
print(df_sentiment_fact.head())

# View results
cursor.execute("SELECT COUNT(*) FROM reddit_sentiment_fact_table_weekly WHERE type = 'post'")
print("Row count:", cursor.fetchone()[0])

# View results
cursor.execute("""
SELECT type, COUNT(*)
FROM reddit_sentiment_fact_table_weekly
GROUP BY type;
""")
print(cursor.fetchall())

# Cleanup
cursor.close()
conn.close()


   id     type company_name keyword keyword_label  year  week sentiment  count
0   1  comment       AbbVie    ABBV  stock_symbol  2016     6  Negative      3
1   2  comment       AbbVie    ABBV  stock_symbol  2016     6   Neutral      3
2   3  comment       AbbVie    ABBV  stock_symbol  2016     6  Positive      4
3   4  comment       AbbVie    ABBV  stock_symbol  2019    51  Negative      4
4   5  comment       AbbVie    ABBV  stock_symbol  2019    51   Neutral      3
Row count: 27683
[('comment', 69562), ('post', 27683)]


In [19]:
# Connect to the MySQL database
conn = mysql.connector.connect(
    host=DB_HOST,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME
)

cursor = conn.cursor()

# Get distinct year-week combinations
cursor.execute("SELECT distinct year, week FROM reddit_sentiment_fact_table_weekly")
print(cursor.fetchall())

# Cleanup
cursor.close()
conn.close()


[(2016, 6), (2019, 51), (2020, 44), (2020, 46), (2020, 49), (2021, 4), (2021, 7), (2021, 8), (2021, 9), (2021, 12), (2021, 14), (2021, 17), (2021, 22), (2021, 23), (2021, 26), (2021, 28), (2021, 29), (2021, 35), (2021, 48), (2021, 50), (2022, 1), (2022, 5), (2022, 9), (2022, 38), (2022, 40), (2023, 5), (2023, 12), (2023, 27), (2023, 28), (2023, 40), (2023, 48), (2023, 52), (2024, 4), (2024, 7), (2024, 8), (2024, 9), (2024, 12), (2024, 13), (2024, 14), (2024, 15), (2024, 23), (2024, 28), (2024, 29), (2024, 30), (2024, 32), (2024, 33), (2024, 36), (2024, 38), (2024, 39), (2024, 40), (2024, 41), (2024, 49), (2024, 50), (2024, 51), (2025, 11), (2016, 40), (2016, 41), (2017, 2), (2017, 3), (2017, 5), (2017, 33), (2017, 52), (2018, 26), (2018, 34), (2018, 38), (2018, 42), (2019, 47), (2020, 4), (2020, 12), (2020, 13), (2020, 17), (2020, 18), (2020, 22), (2020, 29), (2020, 33), (2020, 47), (2020, 50), (2020, 51), (2020, 53), (2021, 3), (2021, 6), (2021, 10), (2021, 13), (2021, 18), (2021, 20)